# SETUP PACKAGES

In [47]:
%pip install google-genai
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# DO NOT CHANGE ANYTHING IN THE CELL BELOW

In [48]:
import os
from google import genai
from google.genai import types
from google import genai
from pydantic import BaseModel, Field
from typing import List
from copy import deepcopy
import json

os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client()

# ABLE TO CHANGE

In [49]:
model_LLM = "gemini-2.5-flash"

PUZZLE_FILE = "../dataset/puzzle_07.json"
with open(PUZZLE_FILE, 'r', encoding='utf-8') as f:
    puzzle = json.load(f)

In [50]:
grid_size = puzzle["grid_size"]
unit_sum = grid_size * (grid_size + 1) // 2
cage_count = len(puzzle["cages"])
total_equations = grid_size * 3 + cage_count

if grid_size == 9:
    box_shape = "3x3"
    box_rows, box_cols = 3, 3
elif grid_size == 6:
    box_shape = "2x3"
    box_rows, box_cols = 2, 3
else:
    raise ValueError(f"Unsupported grid size: {grid_size}")

# SETUP

## Convert cages to text format

In [51]:
cages_text = []
for cage in puzzle['cages']:
    cells = cage['cells']
    cage_sum = cage['sum']
    
    # Convert cell coordinates to text format (1-indexed)
    cell_strs = [f"r{r+1}c{c+1}" for r, c in cells]
    cage_str = "- " + " + ".join(cell_strs) + f" = {cage_sum}"
    cages_text.append(cage_str)

# Print the result
for cage_text in cages_text:
    print(cage_text)

- r1c1 + r2c1 + r3c1 = 10
- r1c2 + r1c3 + r2c2 = 21
- r1c4 + r2c3 + r2c4 = 12
- r1c5 + r1c6 + r2c5 = 15
- r1c7 + r1c8 = 11
- r1c9 + r2c9 = 16
- r2c6 + r2c7 + r2c8 = 10
- r3c2 + r3c3 + r4c3 = 10
- r3c4 + r3c5 + r4c4 = 18
- r3c6 + r3c7 = 9
- r3c8 + r4c8 = 11
- r3c9 + r4c9 = 13
- r4c1 + r5c1 = 12
- r4c2 + r5c2 = 9
- r5c3 + r5c4 = 9
- r4c5 + r5c5 + r6c5 = 7
- r4c6 + r4c7 = 13
- r5c6 + r5c7 = 14
- r5c8 + r6c8 = 14
- r5c9 + r6c9 = 3
- r6c1 + r7c1 = 8
- r8c1 + r9c1 = 15
- r6c2 + r7c2 = 14
- r6c3 + r6c4 = 12
- r7c3 + r7c4 = 6
- r8c2 + r8c3 + r8c4 = 16
- r9c2 + r9c3 = 3
- r6c6 + r7c5 + r7c6 = 20
- r6c7 + r7c7 + r7c8 = 12
- r8c5 + r9c4 + r9c5 = 21
- r8c6 + r8c7 + r9c6 = 12
- r8c8 + r9c7 + r9c8 = 16
- r7c9 + r8c9 + r9c9 = 13


## Convert puzzle grid to markdown table and back

In [52]:
def grid_to_markdown(grid):
    """Convert a 2D list grid to a markdown table string"""
    rows = []
    # Header row
    rows.append("| " + " | ".join([""] * len(grid[0])) + " |")
    # Separator row
    rows.append("|" + "|".join(["---"] * len(grid[0])) + "|")
    # Data rows
    for row in grid:
        rows.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(rows)

def markdown_to_grid(markdown_str):
    """Convert a markdown table string back to a 2D list grid"""
    lines = [line.strip() for line in markdown_str.split("\n") if line.strip()]
    grid = []
    for line in lines[2:]:  # Skip header and separator rows
        # Remove leading and trailing pipes and split by pipe
        cells = [cell.strip() for cell in line.split("|")[1:-1]]
        grid.append([int(cell) for cell in cells if cell])
    return grid

markdown_table = grid_to_markdown(puzzle['puzzle'])
print(markdown_table)

|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |


## Load cheat-sheet

In [53]:
with open('killer_sudoku_cheat_sheet_9x9.md', 'r', encoding='utf-8') as file:
    cheat_sheet = file.read()

# Single Prompt

In [54]:
prompt = """
You are an expert Killer Sudoku solver.

Your task is to solve the Killer Sudoku puzzle based on the rules, reference material, and puzzle data provided below.

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.

Standard Sudoku Rules:
- Each row must contain digits 1 to {grid_size} exactly once.
- Each column must contain digits 1 to {grid_size} exactly once.
- Each subgrid must contain digits 1 to {grid_size} exactly once.

Killer Sudoku Rules:
- The grid is divided into cages.
- The digits in each cage must sum to the cage target.
- Digits cannot repeat within the same cage.

2. Mathematical Facts

For this puzzle:
- Row sum: {unit_sum}
- Column sum: {unit_sum}
- Subgrid sum: {unit_sum}
- Number of cage equations: {cage_count}
- Total sum equations: {total_equations}

3. Reference Material: Killer Sudoku Cheat Sheet

- Combinations are written as digit sequences without separators.
- Example: "12" means {{1, 2}}
- Example: "135" means {{1, 3, 5}}

Cheat sheet:

{cheat_sheet}

4. Puzzle Instance

Initial State:

{puzzle}

Cages:

{cages}

5. Solving Requirements

Solve the puzzle logically using:
- Sudoku row constraints
- Sudoku column constraints
- Sudoku subgrid constraints
- Killer cage sum constraints
- Cage distinctness constraints
- Cheat sheet combinations

Do NOT guess.
Do NOT invent values.
Only return a solution if it is logically consistent with ALL constraints.

6. Mandatory Self-Verification Before Output

Before returning the final solution, silently verify ALL of the following:

Sudoku validation:
- Every row contains digits 1 to {grid_size} exactly once.
- Every column contains digits 1 to {grid_size} exactly once.
- Every subgrid contains digits 1 to {grid_size} exactly once.

Killer Sudoku validation:
- Every cage sums exactly to its target.
- No repeated digit exists inside any cage.

Global validation:
- Every cell is filled.
- No contradictions exist.

If ANY validation fails:
DO NOT return an invalid board.

7. Output Requirement

Return ONLY valid JSON matching this schema:

{{
  "board": [[...]]
}}

If you cannot determine a fully valid solution with certainty, return:

{{
  "board": []
}}

Do not include:
- explanations
- markdown
- comments
- reasoning text
- extra text
"""

In [55]:
class SudokuResult(BaseModel):
    board: List[List[int]] = Field(
        description="The solved Sudoku grid as a 2D list of integers."
    )

In [56]:
# config = types.GenerateContentConfig(
#     response_json_schema=SudokuResult.model_json_schema(),
#     response_mime_type="application/json"
# )

# response = client.models.generate_content(
#     model=model_LLM,
#     contents=prompt.format(
#         grid_size=grid_size,
#         box_shape=box_shape,
#         unit_sum=unit_sum,
#         cage_count=cage_count,
#         total_equations=total_equations,
#         puzzle=markdown_table,
#         cages="\n".join(cages_text),
#         cheat_sheet=cheat_sheet
#     ),
#     config=config
# )

# result = SudokuResult.model_validate_json(response.text)

# print(grid_to_markdown(result.board))

# Multi-step prompt

In [57]:
grid_size = puzzle["grid_size"]
unit_sum = grid_size * (grid_size + 1) // 2
cage_count = len(puzzle["cages"])
total_equations = grid_size * 3 + cage_count

if grid_size == 9:
    box_shape = "3x3"
elif grid_size == 6:
    box_shape = "2x3"
else:
    raise ValueError("Unsupported grid size")

In [58]:
analysis_prompt = """
You are an expert Killer Sudoku reasoning engine.

Your current task is strictly limited to ANALYSIS and NOTE-TAKING.
Do NOT solve the puzzle completely.
Do NOT output a final grid.
Do NOT make guesses.

Your objective is to analyze the current board state as a constraint system and create a logical scratchpad.
This scratchpad will be used in the next step to commit only mathematically justified digits.

[KNOWLEDGE RECAP]

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.
- Standard Sudoku Rule:
  - Each row must contain digits 1 to {grid_size} exactly once.
  - Each column must contain digits 1 to {grid_size} exactly once.
  - Each subgrid must contain digits 1 to {grid_size} exactly once.
- Killer Sudoku Rule:
  - The grid is divided into cages.
  - The digits in each cage must sum to the cage target.
  - Digits cannot repeat within the same cage.
- Important uniqueness rule:
  - Different cells are NOT required to have different values globally.
  - Two cells must be different only if they share a row, a column, a 3x3 box, or the same killer cage.
  - If algebra implies cell A = cell B, this is NOT a contradiction unless A and B are constrained to be different by row, column, box, or cage.
  - Do not reject a deduction only because two cells in different constraints may have the same value.
2. Equation System View

Treat the puzzle as a constraint system.

- Number of row equations: {grid_size}
- Number of column equations: {grid_size}
- Number of subgrid equations: {grid_size}
- Number of cage equations: {cage_count}
- Total sum equations: {total_equations}

Every completed row, column, and subgrid sums to {unit_sum}.

Each equation has two constraints:
1. The values must sum to the target.
2. The values inside the same row, column, subgrid, or cage must be distinct.

3. Reference Material: Cheat Sheet

Important interpretation rules:

- Combinations are written as digit sequences without separators.
- Example: "12" means {{1, 2}}.
- Example: "135" means {{1, 3, 5}}.
- Use the cheat sheet to restrict cage combinations.

Cheat sheet:

{cheat_sheet}

[LAST MOVE LOG]

- Cell Modified: {last_cell}
- Value: {last_value}
- AI's Reasoning: "{last_reasoning}"
- Confidence: {last_is_certain}

[CURRENT STATE]

Current board:

{current_board}

Empty cells you are allowed to reason/fill:

{empty_cells}

IMPORTANT:
- You may ONLY propose cells from this empty_cells list.
- Any cell not in this list is already filled.
- Never output a filled cell as a forced placement.

Cages:

{cages}

[PREVIOUS ANALYSIS]

{previous_analysis}

[INSTRUCTIONS]

1. First, inspect the LAST MOVE LOG.
2. If the last move is not N/A, update all equations affected by that move:
   - its row equation
   - its column equation
   - its 3x3 box equation
   - its cage equation
3. For each affected equation, compute:
   - known values
   - remaining cells
   - remaining sum
   - allowed digits
4. Propagate the updated restrictions to related rows, columns, boxes, and cages.
5. Use the cheat sheet to restrict cage combinations.
6. List candidate eliminations only when they are logically justified.
7. Highlight forced placements only if they are mathematically certain.
8. If no forced placement exists, clearly state NO_CERTAIN_MOVE.
9. Keep the analysis compact. Focus only on affected rows, columns, boxes, cages, and the strongest deductions.
10. Do not enumerate every candidate for every empty cell.
11. Output your analysis exactly following the Markdown template below.

{analysis_template}
"""

In [59]:
fill_prompt = """
You are the Execution Engine of an advanced Killer Sudoku solver.

Your task is to evaluate the provided logical analysis and commit at most ONE valid digit to the board.

Do NOT guess.
Only fill a digit if it is mathematically forced.

[KNOWLEDGE RECAP]

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.
- Standard Sudoku Rule:
  - Each row must contain digits 1 to {grid_size} exactly once.
  - Each column must contain digits 1 to {grid_size} exactly once.
  - Each subgrid must contain digits 1 to {grid_size} exactly once.
- Killer Sudoku Rule:
  - The digits in each cage must sum to the cage target.
  - Digits cannot repeat within the same cage.
- Important uniqueness rule:
  - Different cells are NOT required to have different values globally.
  - Two cells must be different only if they share a row, a column, a 3x3 box, or the same killer cage.
  - If algebra implies cell A = cell B, this is NOT a contradiction unless A and B are constrained to be different by row, column, box, or cage.
  - Do not reject a deduction only because two cells in different constraints may have the same value.

2. Equation System View

- Every completed row, column, and subgrid sums to {unit_sum}.
- Number of cage equations: {cage_count}
- Total equations: {total_equations}

3. Reference Material: Cheat Sheet

{cheat_sheet}

[CURRENT STATE]

Current board:

{current_board}

Empty cells you are allowed to reason/fill:

{empty_cells}

IMPORTANT:
- You may ONLY propose cells from this empty_cells list.
- Any cell not in this list is already filled.
- Never output a filled cell as a forced placement.

Cages:

{cages}

[LOGICAL ANALYSIS]

{current_analysis}

[INSTRUCTIONS]

1. Review the analysis carefully.
2. Choose ONE cell only if its value is mathematically forced.
3. You can ONLY modify cells that currently contain 0.
4. Before choosing a cell, verify from the CURRENT BOARD that the cell contains 0.
5. Never choose a cell that already has a non-zero value.
6. If the analysis suggests filling a non-zero cell, reject that suggestion and return no placement.
7. If no forced cell exists, return no placement.
8. Do NOT make educated guesses.
9. Output strictly as a valid JSON object.

"""

In [60]:
current_board = deepcopy(puzzle['puzzle'])
previous_analysis = "This is the very first turn. Please generate the initial analysis."
last_cell = "N/A"
last_value = "N/A"
last_reasoning = "N/A"
last_is_certain = "N/A"

analysis_template = None
with open('analysis_template_9x9.md', 'r', encoding='utf-8') as file:
    analysis_template = file.read()

def parse_cell(cell_str):
    """
    Parse cell string in format 'rXcY' to (row, col) with 0-indexed coordinates.
    Example: 'r2c3' -> (1, 2)
    """
    # Remove 'r' and 'c', split by 'c'
    parts = cell_str.lower().replace('r', '').split('c')
    row = int(parts[0]) - 1  # Convert to 0-indexed
    col = int(parts[1]) - 1  # Convert to 0-indexed
    return row, col

def update_board(board, cell_str, value):
    """
    Update the board with a new value at the specified cell.
    cell_str: string in format 'rXcY' (e.g., 'r2c3')
    value: integer value to fill
    """
    row, col = parse_cell(cell_str)
    board[row][col] = value
    return board

def validate_current_state(solution, current_board):
    grid_size = len(current_board)

    for r in range(grid_size):
        for c in range(grid_size):
            if current_board[r][c] != 0 and current_board[r][c] != solution[r][c]:
                return False
    return True

def is_all_filled(board):
    for row in board:
        for cell in row:
            if cell == 0:
                return False
    return True

def get_empty_cells(board):
    empty_cells = []
    for r in range(len(board)):
        for c in range(len(board[r])):
            if board[r][c] == 0:
                empty_cells.append(f"r{r+1}c{c+1}")
    return empty_cells

def compact_text(text, max_chars=4000):
    if not text:
        return ""
    if len(text) <= max_chars:
        return text
    return text[-max_chars:]


In [61]:
execution_log = [] 
step_count = 0
final_status = "In Progress"
error_type = "None"

In [62]:
from pydantic import BaseModel, Field
from typing import Optional

class SudokuAnalysisResult(BaseModel):
    analysis: str = Field(
        description="The logical analysis and scratchpad generated by the LLM."
    )

class SudokuFillResult(BaseModel):
    reasoning: str = Field(
        description="Explain why this digit is forced. If no forced move exists, explain why."
    )
    cell: Optional[str] = Field(
        default=None,
        description="The cell being filled, in format rXcY. Use null if no forced move exists."
    )
    value: Optional[int] = Field(
        default=None,
        description="The digit being filled. Use null if no forced move exists."
    )
    is_certain: bool = Field(
        description="True only if the digit is mathematically forced."
    )

In [63]:
def call_llm_for_analysis():
    global previous_analysis

    config = types.GenerateContentConfig(
        temperature=0,
        response_json_schema=SudokuAnalysisResult.model_json_schema(),
        response_mime_type="application/json"
    )
    empty_cells = ", ".join(get_empty_cells(current_board))
    previous_analysis_context = compact_text(previous_analysis, 2500)

    contents = analysis_prompt.format(
        grid_size=grid_size,
        box_shape=box_shape,
        unit_sum=unit_sum,
        cage_count=cage_count,
        total_equations=total_equations,
        cheat_sheet=cheat_sheet,
        last_cell=last_cell,
        last_value=last_value,
        last_reasoning=last_reasoning,
        last_is_certain=last_is_certain,
        current_board=grid_to_markdown(current_board),
        cages="\n".join(cages_text),
        previous_analysis=previous_analysis_context,
        analysis_template=analysis_template,
        empty_cells=empty_cells
    )

    try:
        response = client.models.generate_content(
            model=model_LLM,
            contents=contents,
            config=config
        )
        result = SudokuAnalysisResult.model_validate_json(response.text)
    except Exception as e:
        print(f"Analysis JSON failed, retrying with shorter context: {e}")
        retry_contents = contents + "\n\nCRITICAL RETRY INSTRUCTION: Return a much shorter valid JSON object. Keep the analysis under 1200 words. Do not list all candidates."
        response = client.models.generate_content(
            model=model_LLM,
            contents=retry_contents,
            config=config
        )
        result = SudokuAnalysisResult.model_validate_json(response.text)

    previous_analysis = compact_text(result.analysis, 6000)


In [64]:
def call_llm_for_fill():
    global current_board, last_cell, last_value, last_reasoning, last_is_certain
    global execution_log, step_count
    global previous_analysis, analysis_history

    config = types.GenerateContentConfig(
        temperature=0,
        response_json_schema=SudokuFillResult.model_json_schema(),
        response_mime_type="application/json"
    )
    empty_cells = ", ".join(get_empty_cells(current_board))
    contents = fill_prompt.format(
        grid_size=grid_size,
        box_shape=box_shape,
        unit_sum=unit_sum,
        cage_count=cage_count,
        total_equations=total_equations,
        cheat_sheet=cheat_sheet,
        current_board=grid_to_markdown(current_board),
        cages="\n".join(cages_text),
        current_analysis=compact_text(previous_analysis, 6000),
        empty_cells=empty_cells
    )

    try:
        response = client.models.generate_content(
            model=model_LLM,
            contents=contents,
            config=config
        )
        result = SudokuFillResult.model_validate_json(response.text)
    except Exception as e:
        print(f"Fill JSON failed, retrying with strict JSON instruction: {e}")
        retry_contents = contents + "\n\nCRITICAL RETRY INSTRUCTION: Return ONLY one valid JSON object with keys reasoning, cell, value, is_certain. Use null for cell/value if no forced move exists."
        response = client.models.generate_content(
            model=model_LLM,
            contents=retry_contents,
            config=config
        )
        result = SudokuFillResult.model_validate_json(response.text)

    analysis_summary = (
        previous_analysis[:300] + "..."
        if len(previous_analysis) > 300
        else previous_analysis
    )

    # Case 1: LLM không đưa ra nước chắc chắn
    if result.cell is None or result.value is None or result.is_certain is False:
        last_cell = "N/A"
        last_value = "N/A"
        last_reasoning = result.reasoning
        last_is_certain = False

        step_count += 1
        execution_log.append({
            "step": step_count,
            "analysis_summary": analysis_summary,
            "chosen_cell": None,
            "value": None,
            "reasoning": result.reasoning,
            "is_certain": False,
            "error_type": "No Certain Move",
            "board_state": deepcopy(current_board)
        })

        return False

    row, col = parse_cell(result.cell)

    # Case 2: LLM chọn ô đã có giá trị
    if current_board[row][col] != 0:
        print(f"Cell already filled: {result.cell}")
        print("Reject move and reset analysis history.")

        last_cell = "N/A"
        last_value = "N/A"
        last_reasoning = f"Rejected because {result.cell} is already filled."
        last_is_certain = False

        step_count += 1
        execution_log.append({
            "step": step_count,
            "analysis_summary": analysis_summary,
            "chosen_cell": result.cell,
            "value": result.value,
            "reasoning": last_reasoning,
            "is_certain": False,
            "error_type": "Cell Already Filled",
            "board_state": deepcopy(current_board)
        })

        analysis_history = []
        previous_analysis = ""

        return False

    # Case 3: LLM chọn sai đáp án
    if puzzle["solution"][row][col] != result.value:
        correct_value = puzzle["solution"][row][col]

        print(f"Wrong move detected: {result.cell} = {result.value}")
        print(f"Correct value should be {correct_value}")
        print("Reject move and reset analysis history.")

        last_cell = "N/A"
        last_value = "N/A"
        last_reasoning = f"Rejected wrong move: {result.cell} = {result.value}. Correct value is {correct_value}."
        last_is_certain = False

        step_count += 1
        execution_log.append({
            "step": step_count,
            "analysis_summary": analysis_summary,
            "chosen_cell": result.cell,
            "value": result.value,
            "reasoning": last_reasoning,
            "is_certain": False,
            "error_type": "Wrong Move",
            "correct_value": correct_value,
            "board_state": deepcopy(current_board)
        })

        analysis_history = []
        previous_analysis = ""

        return False

    # Case 4: nước đi hợp lệ
    current_board = update_board(current_board, result.cell, result.value)

    last_cell = result.cell
    last_value = result.value
    last_reasoning = result.reasoning
    last_is_certain = result.is_certain

    step_count += 1
    execution_log.append({
        "step": step_count,
        "analysis_summary": analysis_summary,
        "chosen_cell": result.cell,
        "value": result.value,
        "reasoning": result.reasoning,
        "is_certain": result.is_certain,
        "error_type": "None",
        "board_state": deepcopy(current_board)
    })

    return True

In [65]:
import copy

def fresh_rebuild_from_current_board():
    global puzzle, current_board
    global previous_analysis, last_cell, last_value, last_reasoning, last_is_certain

    puzzle["puzzle"] = copy.deepcopy(current_board)
    current_board = copy.deepcopy(puzzle["puzzle"])

    previous_analysis = ""
    last_cell = "N/A"
    last_value = "N/A"
    last_reasoning = "N/A"
    last_is_certain = "N/A"

    print("Fresh rebuild triggered.")
    print("Now puzzle['puzzle'] and current_board are the same.")

## Loop

In [ ]:
max_no_move_retry = 2

step = 0
no_move_retry = 0

while validate_current_state(puzzle["solution"], current_board) and not is_all_filled(current_board):
    step += 1

    print(f"\n========== STEP {step} ==========")
    print("Current board:")
    print(grid_to_markdown(current_board))

    print("Calling analysis...")
    call_llm_for_analysis()
    print("Analysis done.")

    print("Calling fill...")
    moved = call_llm_for_fill()
    print("Fill done.")

    if moved:
        no_move_retry = 0
        print("Move accepted.")
        print(grid_to_markdown(current_board))
        continue

    no_move_retry += 1
    print(f"No certain move. Retry {no_move_retry}/{max_no_move_retry}")

    if no_move_retry <= max_no_move_retry:
        fresh_rebuild_from_current_board()
        continue

    print("No move after retries. Stopping.")
    break



========== STEP 1 ==========
Current board:
|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
Calling analysis...


In [ ]:
import os
import json

grid_size = puzzle["grid_size"]

is_correct = (current_board == puzzle["solution"])

if is_correct:
    final_status = "Success"
    error_type = "None"

elif not is_all_filled(current_board):
    final_status = "Failed"
    error_type = "No Certain Move"

else:
    final_status = "Failed"
    error_type = "Incorrect Solution"

    for r in range(grid_size):
        for c in range(grid_size):
            if current_board[r][c] != puzzle["solution"][r][c]:
                error_type = "Incorrect Solution"
                break
        if error_type == "Incorrect Solution":
            break


log_data = {
    "puzzle_id": puzzle["id"],
    "difficulty": puzzle["difficulty"],
    "grid_size": puzzle["grid_size"],
    "model": model_LLM,
    "total_steps": step_count,
    "final_status": final_status,
    "error_type": error_type,
    "final_board": current_board,
    "solution": puzzle["solution"],
    "execution_log": execution_log
}


# =========================
# model-based output folder
# =========================
if "lite" in model_LLM.lower():
    model_folder = "lite"
elif "flash" in model_LLM.lower():
    model_folder = "flash"
else:
    model_folder = "pro"

output_dir = os.path.join("outputs", model_folder)
os.makedirs(output_dir, exist_ok=True)

out_file = os.path.join(
    output_dir,
    f"log_puzzle_{puzzle['id']:02d}.json"
)

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(log_data, f, ensure_ascii=False, indent=2)

print(f"Đã lưu log: {out_file}")
print(f"Puzzle #{puzzle['id']} | Độ khó: {puzzle['difficulty']} | Tổng bước: {step_count}")
print(f"Kết quả: {final_status} | Loại lỗi: {error_type}")

Đã lưu log: outputs\lite\log_puzzle_07.json
Puzzle #7 | Độ khó: hard | Tổng bước: 3
Kết quả: Failed | Loại lỗi: No Certain Move


# Validate result

In [ ]:
def validate_board(board, cages):
    grid_size = len(board)
    expected = list(range(1, grid_size + 1))

    if grid_size == 9:
        box_rows, box_cols = 3, 3
    elif grid_size == 6:
        box_rows, box_cols = 2, 3
    else:
        raise ValueError(f"Unsupported grid size: {grid_size}")

    valid = True

    for i, row in enumerate(board):
        if sorted(row) != expected:
            print(f"Row {i + 1} is invalid: {row}")
            valid = False

    for col in range(grid_size):
        column = [board[row][col] for row in range(grid_size)]
        if sorted(column) != expected:
            print(f"Column {col + 1} is invalid: {column}")
            valid = False

    for row_start in range(0, grid_size, box_rows):
        for col_start in range(0, grid_size, box_cols):
            nums = []
            for r in range(row_start, row_start + box_rows):
                for c in range(col_start, col_start + box_cols):
                    nums.append(board[r][c])

            if sorted(nums) != expected:
                print(f"Subgrid ({row_start + 1},{col_start + 1}) is invalid: {nums}")
                valid = False

    for idx, cage in enumerate(cages):
        cells = cage["cells"]
        target_sum = cage["sum"]
        values = [board[r][c] for r, c in cells]

        if sum(values) != target_sum:
            print(
                f"Cage {idx + 1} is invalid: "
                f"cells={cells}, sum={sum(values)}, expected={target_sum}"
            )
            valid = False

        if len(values) != len(set(values)):
            print(
                f"Cage {idx + 1} has repeated digits: "
                f"cells={cells}, values={values}"
            )
            valid = False

    if valid:
        print("Validation complete: board is valid.")
    else:
        print("Validation complete: board has errors.")

    return valid

In [ ]:
validate_board(result.board, puzzle["cages"])

NameError: name 'result' is not defined